In [8]:
import os
from pathlib import Path
import subprocess

# --- base configuration ---
BASE_DIR = Path("/Users/christoffer/work/karolinska/development/oligo-mtDSB")
NOTEBOOK_DIRS = ["notebooks-01", "notebooks-02", "notebooks-03"]
OUTPUT_DIR = BASE_DIR / "exports"

# make sure output dir exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# --- find all notebooks recursively ---
notebooks = []
for nb_dir in NOTEBOOK_DIRS:
    nb_path = BASE_DIR / nb_dir
    if nb_path.exists():
        notebooks.extend(nb_path.rglob("*.ipynb"))

if not notebooks:
    print("⚠️ No notebooks found in specified directories.")
else:
    print(f"📚 Found {len(notebooks)} notebooks to export...\n")

    # --- export each notebook using nbconvert ---
    for nb in notebooks:
        print(f"→ Converting: {nb.relative_to(BASE_DIR)}")
        try:
            subprocess.run(
                [
                    "jupyter", "nbconvert",
                    "--to", "script",
                    "--output-dir", str(OUTPUT_DIR),
                    str(nb)
                ],
                check=True,
                stdout=subprocess.PIPE,
                stderr=subprocess.PIPE,
            )
        except subprocess.CalledProcessError as e:
            print(f"⚠️ Failed: {nb.name}\n{e.stderr.decode()}")

    print(f"\n✅ Done! All .py scripts written to: {OUTPUT_DIR}")

📚 Found 68 notebooks to export...

→ Converting: notebooks-01/15_Allen_ABC_10xv2_concat.ipynb
→ Converting: notebooks-01/00_batch_processing.ipynb
→ Converting: notebooks-01/11_comparing_time_points.ipynb
→ Converting: notebooks-01/34_RBD_subset.ipynb
→ Converting: notebooks-01/26_agexconditionxregion_OLs.ipynb
→ Converting: notebooks-01/22_decoupler.ipynb
→ Converting: notebooks-01/24_pertpy_agexcondition_oligo.ipynb.ipynb
→ Converting: notebooks-01/17_cell_type_annotation.ipynb
→ Converting: notebooks-01/03_sub_cluster_MiGL.ipynb
→ Converting: notebooks-01/10_add_counts.ipynb
→ Converting: notebooks-01/30_run_bayes_cond_OL.ipynb
→ Converting: notebooks-01/31_qr_code_gen.ipynb
→ Converting: notebooks-01/23_pertpy_OL.ipynb
→ Converting: notebooks-01/32_make_presentation_marker_cellT.ipynb
→ Converting: notebooks-01/20_compartment_read_based.ipynb
→ Converting: notebooks-01/28_run_bayes_all_genes.ipynb
→ Converting: notebooks-01/29_run_bayes_for_potential_genes.ipynb
→ Converting: noteb

In [9]:
import os
from pathlib import Path
import ast
import textwrap

BASE_DIR = Path("/Users/christoffer/work/karolinska/development/oligo-mtDSB")
EXPORT_DIR = BASE_DIR / "exports"
UTILS_DIR = BASE_DIR / "oligo_mtDSB_utils"

UTILS_DIR.mkdir(parents=True, exist_ok=True)

def get_functions_from_file(py_path: Path):
    """
    Parse a .py file with ast and return a list of dicts:
    [
      {
        "name": "plot_region_activation_fingerprint_by_age",
        "code": "def plot_region_activation_fingerprint_by_age(...):\n    ...",
        "module_guess": "plotting"  # rough guess we'll fill later
      },
      ...
    ]
    """
    with open(py_path, "r") as f:
        src = f.read()

    # parse AST
    try:
        tree = ast.parse(src)
    except SyntaxError:
        print(f"⚠️ Skipping {py_path.name}: SyntaxError during parse (maybe nbconvert noise).")
        return []

    # for recovering exact source: we’ll split lines and slice
    src_lines = src.splitlines()

    funcs = []
    for node in tree.body:
        if isinstance(node, ast.FunctionDef):
            # get start/end lines
            start = node.lineno - 1  # 0-index
            # node.end_lineno only in py>=3.8+; fallback if missing
            end = getattr(node, "end_lineno", None)
            if end is None:
                # crude fallback: take from start until next top-level def/class
                # find next node after this one
                following_linenos = [
                    n.lineno - 1
                    for n in tree.body
                    if hasattr(n, "lineno") and n.lineno - 1 > start
                ]
                end = min(following_linenos) - 1 if following_linenos else len(src_lines) - 1
            else:
                end = end - 1

            fn_code = "\n".join(src_lines[start:end+1])
            funcs.append(
                {
                    "name": node.name,
                    "code": fn_code,
                    "origin": py_path.name,
                }
            )

    return funcs

# Walk through EXPORT_DIR and gather all functions
all_funcs = []
for py_file in EXPORT_DIR.glob("*.py"):
    these = get_functions_from_file(py_file)
    all_funcs.extend(these)

print(f"Collected {len(all_funcs)} functions total.")
# peek first few
for fmeta in all_funcs[:5]:
    print(fmeta["name"], "from", fmeta["origin"])

Collected 359 functions total.
_as_csr from 30_run_bayes_cond_OL.py
pseudobulk_by_groups from 30_run_bayes_cond_OL.py
normalize_pseudobulk from 30_run_bayes_cond_OL.py
pseudobulk_to_long_with_celltype from 30_run_bayes_cond_OL.py
filter_low_genes from 30_run_bayes_cond_OL.py


In [10]:
all_funcs

[{'name': '_as_csr',
  'code': 'def _as_csr(X):\n    if sp.issparse(X):\n        return X.tocsr()\n    # dense -> csr\n    return sp.csr_matrix(np.asarray(X), copy=False)',
  'origin': '30_run_bayes_cond_OL.py'},
 {'name': 'pseudobulk_by_groups',
  'code': 'def pseudobulk_by_groups(adata, group_cols, layer=None, keep_celltypes=None):\n    """\n    Sum raw counts per (celltype, sample, age, condition).\n    Returns:\n      pb_counts : dense float32 array (n_groups x n_genes)\n      groups_df : dataframe with group labels (n_groups x len(group_cols))\n      var_names : pandas Index of gene names\n    """\n    # 0) pull matrix & obs\n    if layer is None:\n        X = _as_csr(adata.X)\n    else:\n        if layer not in adata.layers:\n            raise ValueError(f"Layer \'{layer}\' not found in adata.layers")\n        X = _as_csr(adata.layers[layer])\n\n    obs = adata.obs.copy()\n\n    # 1) optional filter of cell types\n    if keep_celltypes is not None:\n        obs = obs.loc[obs[CELL

In [11]:
import re

def guess_bucket(func_name: str, func_code: str):
    """
    Heuristic: assign each function to a submodule.
    You can hand-edit these rules later.
    """

    # lowercase helpers for matching
    low = func_name.lower() + " " + func_code.lower()

    # I/O / loading Xenium / AnnData builders
    if any(k in low for k in [
        "xenium", "anndata", "scanpy as sc", "load_", "read_parquet", "concat", "qc_metrics"
    ]):
        return "io_utils"

    # spatial plotting / spatial neighbors / compartment / RBD
    if any(k in low for k in [
        "spatial", "rbd", "compartment", "plot_spatial", "spatial_neighbors",
        "pseudobinned", "domains_by_rbd"
    ]):
        return "spatial_utils"

    # DE summary, per-region summaries, activation fingerprints
    if any(k in low for k in [
        "differential", "log2fc", "lfc", "pval", "activation_fingerprint",
        "region_activity_summary", "summarize_condition_across_regions"
    ]):
        return "de_analysis_utils"

    # UMAP, dotplot wrappers, barh panels, pathway_panels
    if any(k in low for k in [
        "umap", "dotplot", "plot_region_activation_fingerprint_by_age",
        "plot_pathway_panels", "plot_isr_panels", "barh", "plt."
    ]):
        return "plotting_utils"

    # filtering/cleanup helpers / gene filtering
    if any(k in low for k in [
        "filter_", "clean_", "min_expr", "gene_universe", "deduplicate"
    ]):
        return "qc_utils"

    # fallback
    return "misc_utils"


# assign bucket to each function
for fmeta in all_funcs:
    fmeta["bucket"] = guess_bucket(fmeta["name"], fmeta["code"])

# quick sanity check
buckets = {}
for f in all_funcs:
    buckets.setdefault(f["bucket"], 0)
    buckets[f["bucket"]] += 1

print("Bucket counts:")
for b, n in buckets.items():
    print(f"  {b}: {n}")

Bucket counts:
  misc_utils: 137
  io_utils: 85
  de_analysis_utils: 74
  plotting_utils: 22
  spatial_utils: 36
  qc_utils: 5


In [12]:
from collections import OrderedDict, defaultdict

# group by bucket
bucket_to_funcs = defaultdict(list)
for fmeta in all_funcs:
    bucket_to_funcs[fmeta["bucket"]].append(fmeta)

for bucket, funcs in bucket_to_funcs.items():
    # resolve duplicates by function name
    unique_funcs = OrderedDict()
    for fm in funcs:
        if fm["name"] not in unique_funcs:
            unique_funcs[fm["name"]] = fm
        else:
            # already saw a function with this name.
            # You could diff or choose the longer one.
            old = unique_funcs[fm["name"]]
            if len(fm["code"]) > len(old["code"]):
                unique_funcs[fm["name"]] = fm

    # build module text
    header = [
        '"""',
        f"Auto-generated utilities for {bucket}.",
        "Do not edit by hand without moving changes back into notebooks.",
        "",
        "Each function below was extracted from exported analysis notebooks.",
        '"""',
        "",
        "from typing import *",
        "import numpy as np",
        "import pandas as pd",
        "import scanpy as sc",
        "import anndata as ad",
        "import matplotlib.pyplot as plt",
        "from scipy import sparse",
        "",
    ]

    body_parts = []
    for fname, fm in unique_funcs.items():
        # ensure we have exactly one blank line before each def
        code_block = fm["code"].strip("\n")
        body_parts.append(code_block + "\n")

    module_text = "\n".join(header + body_parts)

    # write file
    out_path = UTILS_DIR / f"{bucket}.py"
    with open(out_path, "w") as f:
        f.write(module_text)

    print(f"Wrote {out_path} with {len(unique_funcs)} functions.")

Wrote /Users/christoffer/work/karolinska/development/oligo-mtDSB/oligo_mtDSB_utils/misc_utils.py with 57 functions.
Wrote /Users/christoffer/work/karolinska/development/oligo-mtDSB/oligo_mtDSB_utils/io_utils.py with 43 functions.
Wrote /Users/christoffer/work/karolinska/development/oligo-mtDSB/oligo_mtDSB_utils/de_analysis_utils.py with 32 functions.
Wrote /Users/christoffer/work/karolinska/development/oligo-mtDSB/oligo_mtDSB_utils/plotting_utils.py with 16 functions.
Wrote /Users/christoffer/work/karolinska/development/oligo-mtDSB/oligo_mtDSB_utils/spatial_utils.py with 10 functions.
Wrote /Users/christoffer/work/karolinska/development/oligo-mtDSB/oligo_mtDSB_utils/qc_utils.py with 3 functions.
